<a href="https://colab.research.google.com/github/Abhishek-S0111/Pipeline-to-Gather-Audio-Transcript-Datasets-from-Youtube/blob/main/Helper_Notebook_to_download_Audio_from_YT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
%%capture
!pip install yt-dlp -U
# !apt-get update
!apt-get install -y ffmpeg

import pandas as pd
import yt_dlp

In [4]:
# %%capture #Comment out if you dont want any outputs
def YT_Link_Generator(VIDEO_ID :str) -> str:
    """Generator Function to generate YT_Link from VIDEO_ID"""
    return "https://www.youtube.com/watch?v=" + VIDEO_ID

def YT_VIDEO_ID_generator(VIDEO_LINK:str) -> str:
    """Generator function to generate VIDEO_IDS from Link"""
    return VIDEO_LINK[-11:]

def Download_Audio(video_ID):
    import yt_dlp # Import yt_dlp inside the function for multiprocessing

    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
            'preferredquality': '192',
        }],
        'outtmpl': f'/content/KrishiDarshan/{video_ID}/audio.%(ext)s'
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([YT_Link_Generator(video_ID)])

In [13]:
links = ["2ELe4jT7G3Y"]

for id in links:
    Download_Audio(id)

[youtube] Extracting URL: https://www.youtube.com/watch?v=2ELe4jT7G3Y
[youtube] 2ELe4jT7G3Y: Downloading webpage


[youtube] 2ELe4jT7G3Y: Downloading android vr player API JSON
[info] 2ELe4jT7G3Y: Downloading 1 format(s): 140
[download] Destination: /content/KrishiDarshan/2ELe4jT7G3Y/audio.m4a
[download] 100% of   22.55MiB in 00:00:01 at 12.50MiB/s  
[FixupM4a] Correcting container of "/content/KrishiDarshan/2ELe4jT7G3Y/audio.m4a"
[ExtractAudio] Destination: /content/KrishiDarshan/2ELe4jT7G3Y/audio.wav
Deleting original file /content/KrishiDarshan/2ELe4jT7G3Y/audio.m4a (pass -k to keep)


In [18]:
!pip install -q google-genai

from google import genai
from google.colab import userdata

# Initialize the unified Gemini client
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

# Test the connection using gemini-3.5-flash
response = client.models.generate_content(
    model='gemini-3.5-flash',
    contents='Connection successful.'
)
print(response.text)

KeyboardInterrupt: 

In [19]:
import time

# 1. Path to your downloaded audio file
file_path = '/content/KrishiDarshan/2ELe4jT7G3Y/audio.wav'

print("Uploading audio file to Gemini...")
audio_file = client.files.upload(file=file_path)

# Wait briefly for the file API to finish processing the state
print("Waiting for file processing...")
time.sleep(3)

print("\n--- Generating Transcript ---")
# 2. Generate the initial transcription
transcription_response = client.models.generate_content(
    model='gemini-3.5-flash',
    contents=[audio_file, "Provide a highly accurate, clean transcript of this audio file. If there are multiple speakers, try to separate them."]
)
print(transcription_response.text)

print("\n--- Starting Conversation Mode ---")
# 3. Create an ongoing chat session with the audio context
chat = client.chats.create(model='gemini-3.5-flash')

# Initialize the chat with the audio file context
chat.send_message(contents=[audio_file, "Analyze this audio file. I will now ask you questions about its contents."])

print("Chat initialized! You can now ask questions about the audio below. (Type 'exit' to stop)")

# 4. Interactive loop to converse with the audio
while True:
    user_input = input("\nYour Question: ")
    if user_input.lower() == 'exit':
        print("Ending chat session.")
        break

    if user_input.strip() == '':
        continue

    response = chat.send_message(user_input)
    print(f"\nGemini: {response.text}")

Uploading audio file to Gemini...
Waiting for file processing...

--- Generating Transcript ---
**कार्यक्रम प्रस्तोता (महिला):** नमस्कार, आप सभी किसान भाई बहनों का स्वागत है कार्यक्रम कृषि दर्शन में। किसान भाइयों, अच्छी फसल हो इसके लिए आप खेतों में दिन रात कड़ी मेहनत करते हैं और आपकी इस मेहनत में आपके सहयोगी हैं मधुमक्खियां। जी हां, मधुमक्खियां करती हैं खेतों में पोलन क्रिया, जिससे फसल अच्छी होती है और किसानों को मिलता है ज्यादा लाभ। बहुत से किसान भाई बहन ऐसे हैं जो केवल मधुमक्खी पालन करके अच्छा लाभ ले रहे हैं। मधुमक्खियां फूलों से रस लाती हैं और किसान भाई-बहन प्रोसेसिंग कर शहद का उत्पादन करते हैं। प्रोसेसिंग के दौरान कोई भी चीज बेकार नहीं जाती। शहद का इस्तेमाल हम खाने के लिए करते हैं तो जो अतिरिक्त वैक्स बचता है उसका इस्तेमाल विभिन्न प्रकार की औषधियों और कॉस्मेटिक को बनाने में किया जाता है। आज हम जानकारी लेंगे मधुमक्खी पालन और उससे संबंधित प्रोसेसिंग के बारे में।

**सूत्रधार (पुरुष):** इस धरती पर हर जीव का अपना महत्व है। उसकी अपनी भूमिका है। इसी वजह से हमारा इको सिस्टम ठीक बना रहता है

TypeError: Chat.send_message() got an unexpected keyword argument 'contents'